<a href="https://colab.research.google.com/github/memo124/Tareas---KODIGO/blob/main/welcome_to_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 0: Agregar importaciones de librerias

In [6]:
from abc import ABC, abstractmethod
from pathlib import Path
import logging

import pandas as pd
import matplotlib.pyplot as plt

## Paso 1: Clase Abstracta

In [7]:
class DataComponent(ABC):
  @abstractmethod
  def execute():
    pass

## Paso 2: Mixins (Logger y Validator)

In [8]:
logging.basicConfig(
    level = logging.INFO,
    format = "[%(asctime)s] [%(levelname)s - %(name)s] %(message)s"
)

class LoggerMixin:
    _logger: logging.Logger | None = None
    
    @property
    def logger(self) -> logging.Logger:
        return self._logger or logging.getLogger(type(self).__name__)
    
    def log_info(self, message):
        self.logger.info(message)

    def log_warning(self, message):
        self.logger.warning(message)

    def log_error(self, message):
        self.logger.error(message)
        
class ValidatorMixin:
    
    def validate_file(self, file_path):
        # Verify that the path corresponds to an existing file
        if not Path(file_path).is_file():
            raise FileNotFoundError(
                f"No existe el archivo: {file_path}"
            )
            
    def validate_columns(self, dataframe, required_columns):
        # Find the required columns that do not exist in the DataFrame
        missing_columns = [
            column
            for column in required_columns
            if column not in dataframe.columns
        ]

        # Stops the process if at least one column is missing
        if missing_columns:
            raise ValueError(
                f"Faltan las columnas: {missing_columns}"
            )
            
    def validate_nulls(self, dataframe, columns=None):
        columns = columns or dataframe.columns
        
        null_counts = dataframe[columns].isna().sum()
        columns_with_nulls = null_counts[null_counts > 0]

        # Stops the process if null values ​​are found
        if not columns_with_nulls.empty:
            raise ValueError(
                f"Columnas con valores nulos:\n{columns_with_nulls}"
            )

## Paso 3: Clase DataLoader

In [ ]:
class DataLoader(DataComponent, ValidatorMixin, LoggerMixin):
    def __init__(self, file_path):
        self.__file_path = file_path
        
    def execute(self):
        self.log_info(START_LOAD_MSG.format(self.__file_path))
        self.validate_file_exists(self.__file_path)
        
        df = pd.read_csv(self.__file_path)
        self.log_info(SUCCESS_LOAD_MSG.format(len(df)))
        return df

## Paso 4: Clase DataCleaner

In [ ]:
class DataCleaner(DataComponent, ValidatorMixin, LoggerMixin):
    #Cleaning component: infers types and imputes missing values.
    def __init__(self, dataframe, required_columns=None, categorical_fill=None):
        #Stores a private copy of the data and optional cleaning settings
        self.__dataframe = dataframe.copy()
        self.__required_columns = required_columns
        self.__categorical_fill = categorical_fill

    def execute(self):
        #Runs the full cleaning flow and returns the clean DataFrame.
        try:
            self.log_info("Iniciando limpieza de datos")
            if self.__required_columns:
                self.validate_columns(self.__dataframe, self.__required_columns)
                self.log_info("Validación de columnas completada")

            self.__convert_types()
            self.__impute_nulls()
            self.validate_nulls(self.__dataframe)
            self.log_info("Limpieza de datos completada")
            return self.__dataframe
        except (TypeError, ValueError, IndexError) as e:
            self.log_error(f"Error durante la limpieza de datos: {e}")
            raise

    def __convert_types(self):
        #Converts text columns to numeric when most values qualify.
        self.log_info("Convirtiendo tipos de datos")
        for column in self.__dataframe.columns:
            try:
                col = self.__dataframe[column]
                if col.dtype != "object":
                    continue
                sample = col.dropna()
                if sample.empty:
                    continue
                as_numeric = pd.to_numeric(sample, errors='coerce')
                if as_numeric.notna().mean() >= 0.8:
                    self.__dataframe[column] = pd.to_numeric(col, errors='coerce')
                    self.log_info(f"Columna '{column}' convertida a numérica")
            except (TypeError, ValueError, IndexError) as e:
                self.log_warning(f"No se pudo convertir la columna '{column}': {e}")
                continue

    def __impute_nulls(self):
        #Fills missing values: median, configured value, or mode.
        self.log_info("Imputando valores nulos")
        for column in self.__dataframe.columns:
            try:
                null_count = self.__dataframe[column].isna().sum()
                if null_count == 0:
                    continue
                if pd.api.types.is_numeric_dtype(self.__dataframe[column]):
                   fill_value = self.__dataframe[column].median()
                   strategy = "mediana"
                elif self.__categorical_fill is not None:
                    fill_value = self.__categorical_fill
                    strategy = f"valor proporcionado: {self.__categorical_fill}"
                else:
                    fill_value = self.__dataframe[column].mode()[0]
                    strategy = "moda"

                self.__dataframe[column] = self.__dataframe[column].fillna(fill_value)
                self.log_warning(f"Columna '{column}': {null_count} valores nulos imputados usando {strategy}")
            except (TypeError, ValueError, IndexError) as e:
                self.log_warning(f"No se pudo imputar la columna '{column}': {e}")
                continue

    @property
    def summary(self):
        #Read-only descriptive statistics of the dataset.
        return self.__dataframe.describe()

    @property
    def null_report(self):
        #Read-only null count per column.
        return self.__dataframe.isna().sum()